# Stepper Motor

*Written by Grace Lo, Rachel Yan, & Henry Sigel*

The 12V rated [NEMA-17](https://www.adafruit.com/product/324) Stepper Motor is used to control the mechanical design of the flux chamber by opening and closing the chamber. It uses [Adafruit's TMC2209](https://www.adafruit.com/product/6121?srsltid=AfmBOooewzKw0BhmeOOv3fSTSSEoolT53hRkdlx3yQKAozX7bzYRfrfr) Stepper Motor Driver Breakout Board to control the direction, step, and current flow. The current flow is used for the motor's holding torque and without it, the motor's position can shift . But since it is attached to a threaded metal rod, the change is negligible. Therefore, when the motor is not in use, the current flow is disabled to reduce power consumption and prevent the motor from overheating. 

## Pin Connections

Breadboarding best practices should be followed. 

<img src="img/pin_connections.png" alt="Pin Connections" width="400"/>

Pin connections from the **motor driver &rarr; Pico** are as follows:
- VDD &rarr; 3V3(OUT) (pin 36)
- GND &rarr; GND
- DIR &rarr; GPIO 16 (pin 21)
- STEP &rarr; GPIO 17 (pin 22)
- EN &rarr; GPIO 18 (pin 24)

And on the terminal block: 
- ⊕ &rarr; motor input voltage (12V)
- ⊖ &rarr; GND
- 1A/B &rarr; red/yellow
- 2A/B &rarr; green/gray

## Code

All code is in the [CornellFluxChamber](https://github.com/CornellFluxChamber/ChamberCode.git) Github repository. The code for running the stepper motor is incorporated into `flux_chamber.c`.

### Includes

The first lines of code in the C source file include header files. *Don't forget to link these in the CMakeLists.txt file!*

The following files are required to enable the GPIO pins for the motor driver. 

```
#include <stdio.h>
#include "pico/stdlib.h"
```

### Initializations

The C source file initializes the GPIO's as outputs. In addition, it immediately disables the motor using the enable pin to cut the current. 

```
gpio_init(18);              // enable pin
gpio_set_dir(18, GPIO_OUT);
gpio_init(17);              // step pin
gpio_set_dir(17, GPIO_OUT);
gpio_init(16);              // direction pin
gpio_set_dir(16, GPIO_OUT);
gpio_put(18, 1);            // disable motor when not in use
```

### Running the Motor
The motor code is wrapped in a `move_motor` function that takes in arguments for the direction and number of steps the motor should take, on the order of thousands. It first enables current to the motor and defines the direction (0 for clockwise/up, 1 for counterclockwise/down). Then, the STEP pin is toggled for the amount of `steps` defined, which visibly is the length of time for the motor movement. Note that the for loop keeps toggling the STEP pin because the stepper motor is triggered by rising edges, so one toggle means one step. The `sleep_us(1000)` delay between toggles ensures the motor driver has sufficient response time for each step, which in this case is the maximum frequency of 1/1000us.

```
// Function to move motor - 0 is cw up, 1 is ccw down
void move_motor(int direction, int steps)
{
    gpio_put(18, 0);            // enable motor
    gpio_put(16, direction);
    for (int i = 0; i < steps; i++)
    {
        gpio_put(17, true);
        sleep_us(1000);
        gpio_put(17, false);
        sleep_us(1000);
    }
    gpio_put(18, 1);            // disable motor
}
```